In [ ]:
# CELL 1 — install
!pip install -q -U ultralytics
!test -d /content/yolov5 || git clone -q https://github.com/ultralytics/yolov5 /content/yolov5
!pip install -q -r /content/yolov5/requirements.txt
import torch, tensorflow as tf
print("torch", torch.__version__, "| TF", tf.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (Runtime > Change runtime type > GPU)")

torch 2.14.1+cu130 | TF 2.21.0 | GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# CELL 2 — config + Drive
import os, json, time, shutil, random, subprocess, pickle, glob
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

ROOT   = '/content/drive/MyDrive/waste_experiments'      # all results live here
RAW    = '/content/taco_raw'                              # downloaded raw images
YDIR   = '/content/taco_yolo'                             # YOLO-format dataset
CDIR   = '/content/taco_cls'                              # Inception crop dataset
RUNS   = '/content/runs'                                  # local training output (copied to Drive after each run)
for d in (ROOT, RAW, YDIR, CDIR, RUNS, f'{ROOT}/data_info', f'{ROOT}/comparison'): os.makedirs(d, exist_ok=True)

# ---- paper settings ----
IMG, BATCH, EPOCHS = 416, 16, 100            # detector
CLS_IMG, CLS_BATCH, CLS_EPOCHS = 224, 64, 100  # classifier
SEED, SPLIT = 42, (0.8, 0.1, 0.1)
USE_UNOFFICIAL = False        # False = official TACO annotations.json (1500 imgs); True = also unofficial annotations
HYBRID_SCORE = 'det'          # 'det' = detector confidence ; 'det*cls' = detector conf x Inception prob
WORKERS = 8

SUPER = ['cardboard', 'paper', 'plastic', 'metal', 'glass', 'trash']   # paper's 6 categories
# Table 1 of the paper: (TACO category id, super-category)
TACO_SEL = [(4,'plastic'),(5,'plastic'),(29,'plastic'),(36,'plastic'),(39,'plastic'),
            (6,'glass'),(23,'glass'),(26,'glass'),
            (12,'metal'),(52,'metal'),(8,'metal'),(28,'metal'),
            (14,'cardboard'),(16,'cardboard'),(17,'cardboard'),(18,'cardboard'),
            (31,'paper'),(32,'paper'),(33,'paper'),(34,'paper'),
            (25,'trash'),(1,'trash')]
PAPER_COUNTS = {4:50,5:285,29:273,36:451,39:260,6:104,23:6,26:6,12:229,52:20,8:80,28:10,
                14:93,16:45,17:64,18:30,31:42,32:12,33:82,34:27,25:8,1:2}
ID2IDX  = {tid: i for i, (tid, _) in enumerate(TACO_SEL)}           # TACO id -> 0..21
SUP22   = np.array([SUPER.index(s) for _, s in TACO_SEL])           # 22-class idx -> 6-class idx
print(len(TACO_SEL), "detector classes ->", len(SUPER), "super-categories")

Mounted at /content/drive
22 detector classes -> 6 super-categories


In [ ]:
# CELL 3 — download TACO (annotations + images), nothing else is used
import requests
from concurrent.futures import ThreadPoolExecutor
ANN_URL = 'https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json'
ann = requests.get(ANN_URL, timeout=60).json()
if USE_UNOFFICIAL:
    un = requests.get(ANN_URL.replace('annotations.json', 'annotations_unofficial.json'), timeout=60).json()
    off = max(i['id'] for i in ann['images']) + 1; offa = max(a['id'] for a in ann['annotations']) + 1
    for i in un['images']: i['id'] += off
    for a in un['annotations']: a['image_id'] += off; a['id'] += offa
    ann['images'] += un['images']; ann['annotations'] += un['annotations']
print(len(ann['images']), "images,", len(ann['annotations']), "annotations")

def fetch(img):
    out = f"{RAW}/{img['file_name'].replace('/', '_')}"
    if os.path.exists(out) and os.path.getsize(out) > 1000: return True
    for url in (img.get('flickr_640_url'), img.get('flickr_url')):
        if not url: continue
        for _ in range(3):
            try:
                r = requests.get(url, timeout=30)
                if r.status_code == 200 and len(r.content) > 1000:
                    open(out, 'wb').write(r.content); return True
            except Exception: time.sleep(1)
    return False
with ThreadPoolExecutor(32) as ex: ok = list(ex.map(fetch, ann['images']))
print(f"downloaded {sum(ok)}/{len(ok)} images")

1500 images, 4784 annotations
downloaded 1500/1500 images


In [ ]:
# CELL 4 — build identical train/val/test splits + YOLO labels + Inception crops (+ audit vs paper Table 1)
from PIL import Image, ImageOps
cats = {c['id']: c['name'] for c in ann['categories']}
print("\nCategory-name check against paper Table 1:")
for tid, s in TACO_SEL: print(f"  id {tid:>2} -> {cats[tid]:<22} [{s}]")

sel = set(ID2IDX)
by_img = {}
for a in ann['annotations']:
    if a['category_id'] in sel: by_img.setdefault(a['image_id'], []).append(a)
imgs = {i['id']: i for i in ann['images']}
usable = sorted(i for i in by_img if os.path.exists(f"{RAW}/{imgs[i]['file_name'].replace('/', '_')}"))
rng = random.Random(SEED); rng.shuffle(usable)
n = len(usable); n_tr, n_va = int(SPLIT[0]*n), int(SPLIT[1]*n)
split_of = {i: 'train' for i in usable[:n_tr]}
split_of.update({i: 'val' for i in usable[n_tr:n_tr+n_va]}); split_of.update({i: 'test' for i in usable[n_tr+n_va:]})
print(f"\nimages with selected categories: {n}  (train {n_tr} / val {n_va} / test {n-n_tr-n_va})")

def load_fixed(path, W, H):                      # choose EXIF-rotated or raw version, whichever matches annotation aspect
    im = Image.open(path); t = ImageOps.exif_transpose(im).convert('RGB'); r = im.convert('RGB')
    return min([t, r], key=lambda x: abs(x.width / x.height - W / H))

for sp in ('train', 'val', 'test'):
    for sub in ('images', 'labels'): os.makedirs(f'{YDIR}/{sub}/{sp}', exist_ok=True)
    for s in SUPER: os.makedirs(f'{CDIR}/{sp}/{s}', exist_ok=True)

counts22 = {tid: 0 for tid in ID2IDX}; n_crops = {sp: 0 for sp in ('train', 'val', 'test')}
for iid in usable:
    sp, info = split_of[iid], imgs[iid]; W, H = info['width'], info['height']
    im = load_fixed(f"{RAW}/{info['file_name'].replace('/', '_')}", W, H)
    stem = f"{iid:05d}"; im.save(f'{YDIR}/images/{sp}/{stem}.jpg', quality=95)
    sx, sy = im.width / W, im.height / H
    lines = []
    for a in by_img[iid]:
        x, y, w, h = a['bbox']
        if w < 2 or h < 2: continue
        k = ID2IDX[a['category_id']]; counts22[a['category_id']] += 1
        xc, yc = min(max((x + w/2)/W, 0), 1), min(max((y + h/2)/H, 0), 1)
        lines.append(f"{k} {xc:.6f} {yc:.6f} {min(w/W,1):.6f} {min(h/H,1):.6f}")
        if w*sx >= 8 and h*sy >= 8:               # classifier crop labelled with its super-category
            im.crop((int(x*sx), int(y*sy), int((x+w)*sx), int((y+h)*sy))).save(f"{CDIR}/{sp}/{SUPER[SUP22[k]]}/{stem}_{a['id']}.jpg", quality=95)
            n_crops[sp] += 1
    open(f'{YDIR}/labels/{sp}/{stem}.txt', 'w').write('\n'.join(lines))

names = {i: cats[tid] for i, (tid, _) in enumerate(TACO_SEL)}
open(f'{YDIR}/taco.yaml', 'w').write(f"path: {YDIR}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 22\nnames:\n" +
                                      ''.join(f"  {i}: {n_}\n" for i, n_ in names.items()))
print("\ninstances per class (ours vs paper Table 1):")
for tid, s in TACO_SEL: print(f"  {tid:>2} {cats[tid]:<22} ours={counts22[tid]:>4}  paper={PAPER_COUNTS[tid]:>4}")
print("classifier crops:", n_crops)
json.dump({'split': {str(k): v for k, v in split_of.items()}, 'counts22': counts22, 'crops': n_crops,
           'classes22': names, 'super': SUPER}, open(f'{ROOT}/data_info/split_and_counts.json', 'w'), indent=1)


Category-name check against paper Table 1:
  id  4 -> Other plastic bottle   [plastic]
  id  5 -> Clear plastic bottle   [plastic]
  id 29 -> Other plastic          [plastic]
  id 36 -> Plastic film           [plastic]
  id 39 -> Other plastic wrapper  [plastic]
  id  6 -> Glass bottle           [glass]
  id 23 -> Glass cup              [glass]
  id 26 -> Glass jar              [glass]
  id 12 -> Drink can              [metal]
  id 52 -> Scrap metal            [metal]
  id  8 -> Metal bottle cap       [metal]
  id 28 -> Metal lid              [metal]
  id 14 -> Other carton           [cardboard]
  id 16 -> Drink carton           [cardboard]
  id 17 -> Corrugated carton      [cardboard]
  id 18 -> Meal carton            [cardboard]
  id 31 -> Tissues                [paper]
  id 32 -> Wrapping paper         [paper]
  id 33 -> Normal paper           [paper]
  id 34 -> Paper bag              [paper]
  id 25 -> Food waste             [trash]
  id  1 -> Battery                [trash]

image

In [ ]:
# CELL 5 — shared helpers: training runner, predictors, ONE evaluator used for all 8 experiments
import cv2, torch, torchvision
EXPS = {'yolov5s': dict(id=1, fam='v5', w='yolov5s.pt'), 'yolov5l': dict(id=2, fam='v5', w='yolov5l.pt'),
        'yolov8s': dict(id=3, fam='ul', w='yolov8s.pt'), 'yolo26s': dict(id=4, fam='ul', w='yolo26s.pt')}
exp_dir = lambda k: f"{ROOT}/exp{EXPS[k]['id']}_{k}"
hyb_dir = lambda k: f"{ROOT}/exp{EXPS[k]['id']+4}_{k}_inception"
YAML = f'{YDIR}/taco.yaml'
env = dict(os.environ, WANDB_MODE='disabled', COMET_MODE='disabled')

def run_cmd(cmd, log, cwd=None):
    with open(log, 'w') as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd, env=env, bufsize=1)
        for line in p.stdout:
            f.write(line)
            if ' all ' in line or 'summary' in line.lower(): print(line.rstrip()[:160])
        p.wait()
    assert p.returncode == 0, f"command failed, see {log}"

def train_detector(key):
    e, name = EXPS[key], f"exp{EXPS[key]['id']}_{key}"
    if os.path.exists(f"{exp_dir(key)}/weights/best.pt"): print(f"[skip] {name} already trained"); return
    t0 = time.time()
    if e['fam'] == 'v5':
        cmd = ['python', 'train.py', '--img', str(IMG), '--batch', str(BATCH), '--epochs', str(EPOCHS), '--data', YAML,
               '--weights', e['w'], '--project', RUNS, '--name', name, '--exist-ok', '--workers', str(WORKERS),
               '--cache', 'ram', '--seed', '0', '--patience', str(EPOCHS)]
        run_cmd(cmd, f'{RUNS}/{name}.log', cwd='/content/yolov5')
    else:
        cmd = ['yolo', 'detect', 'train', f"model={e['w']}", f'data={YAML}', f'epochs={EPOCHS}', f'imgsz={IMG}', f'batch={BATCH}',
               f'workers={WORKERS}', 'cache=ram', 'seed=0', f'patience={EPOCHS}', 'deterministic=False',
               f'project={RUNS}', f'name={name}', 'exist_ok=True']
        run_cmd(cmd, f'{RUNS}/{name}.log')
    mins = (time.time() - t0) / 60
    shutil.copytree(f'{RUNS}/{name}', exp_dir(key), dirs_exist_ok=True)
    shutil.copy(f'{RUNS}/{name}.log', f'{exp_dir(key)}/train.log')
    json.dump({'train_minutes': mins}, open(f'{exp_dir(key)}/train_time.json', 'w'))
    print(f"[done] {name} in {mins:.1f} min")

# ---------- predictors (conf 0.001, NMS IoU 0.6, max 100 dets, 416 input — same for every model) ----------
TEST_IMGS = lambda: sorted(glob.glob(f'{YDIR}/images/test/*.jpg'))
def predict(key, paths):
    w, out = f"{exp_dir(key)}/weights/best.pt", []
    if EXPS[key]['fam'] == 'v5':
        m = torch.hub.load('/content/yolov5', 'custom', path=w, source='local', verbose=False)
        m.conf, m.iou, m.max_det = 0.001, 0.6, 100
        for i in range(0, len(paths), 16):
            out += [r.cpu().numpy() for r in m(paths[i:i+16], size=IMG).xyxy]
    else:
        from ultralytics import YOLO
        m = YOLO(w)
        for r in m.predict(paths, imgsz=IMG, conf=0.001, iou=0.6, max_det=100, batch=16, verbose=False, stream=True):
            b = r.boxes; out.append(np.c_[b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()] if len(b) else np.zeros((0, 6)))
    return out

def load_gt(paths):
    gts = []
    for p in paths:
        W, H = Image.open(p).size
        lf = p.replace('/images/', '/labels/').replace('.jpg', '.txt'); rows = []
        if os.path.exists(lf):
            for l in open(lf).read().split('\n'):
                if l.strip():
                    c, xc, yc, w, h = l.split(); xc, yc, w, h = float(xc)*W, float(yc)*H, float(w)*W, float(h)*H
                    rows.append([int(c), xc-w/2, yc-h/2, xc+w/2, yc+h/2])
        gts.append(np.array(rows).reshape(-1, 5))
    return gts

def box_iou(a, b):
    ix1, iy1 = np.maximum(a[:, None, 0], b[None, :, 0]), np.maximum(a[:, None, 1], b[None, :, 1])
    ix2, iy2 = np.minimum(a[:, None, 2], b[None, :, 2]), np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2-ix1, 0, None) * np.clip(iy2-iy1, 0, None)
    aa, ab = (a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]), (b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)

_trapz = getattr(np, 'trapezoid', None) or np.trapz
def compute_metrics(preds, gts, nc):
    """P, R (at max-mean-F1 conf, as in YOLO), mAP@.5, mAP@.5:.95 — COCO 101-point AP. preds:[(n,6) x1y1x2y2 conf cls]  gts:[(m,5) cls x1y1x2y2]"""
    thrs = np.linspace(0.5, 0.95, 10); rec = [[] for _ in range(nc)]   # per class: (conf, tp[10])
    n_gt = np.zeros(nc)
    for p, g in zip(preds, gts):
        for c in range(nc):
            gc = g[g[:, 0] == c][:, 1:] if len(g) else np.zeros((0, 4)); n_gt[c] += len(gc)
            pc = p[p[:, 5] == c] if len(p) else np.zeros((0, 6))
            if not len(pc): continue
            pc = pc[np.argsort(-pc[:, 4])]; tp = np.zeros((len(pc), 10), bool)
            if len(gc):
                iou = box_iou(pc[:, :4], gc)
                for ti, t in enumerate(thrs):
                    used = np.zeros(len(gc), bool)
                    for i in range(len(pc)):
                        cand = np.where(used, -1, iou[i]); j = cand.argmax()
                        if cand[j] >= t: used[j] = True; tp[i, ti] = True
            rec[c] += [(pc[i, 4], tp[i]) for i in range(len(pc))]
    ap = np.zeros((nc, 10)); px = np.linspace(0, 1, 1000); P, R = [], []
    valid = [c for c in range(nc) if n_gt[c] > 0]
    for c in valid:
        if not rec[c]: P.append(np.zeros(1000)); R.append(np.zeros(1000)); continue
        conf = np.array([r[0] for r in rec[c]]); tp = np.array([r[1] for r in rec[c]], float)
        o = np.argsort(-conf); conf, tp = conf[o], tp[o]
        for ti in range(10):
            tpc = np.cumsum(tp[:, ti]); fpc = np.cumsum(1 - tp[:, ti])
            r_, p_ = tpc / n_gt[c], tpc / (tpc + fpc + 1e-16)
            mrec, mpre = np.r_[0, r_, 1], np.r_[1, p_, 0]
            mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
            x = np.linspace(0, 1, 101); ap[c, ti] = _trapz(np.interp(x, mrec, mpre), x)
            if ti == 0: P.append(np.interp(-px, -conf, p_, left=0)); R.append(np.interp(-px, -conf, r_, left=0))
    P, R = np.mean(P, 0), np.mean(R, 0); F1 = 2*P*R / (P+R+1e-16); i = F1.argmax()
    return dict(P=float(P[i]), R=float(R[i]), mAP50=float(ap[valid, 0].mean()), mAP50_95=float(ap[valid].mean()),
                n_images=len(preds), n_gt=int(n_gt.sum()), classes_evaluated=len(valid),
                per_class_AP50={int(c): float(ap[c, 0]) for c in valid})

def get_detections(key):
    pk = f"{exp_dir(key)}/test_detections.pkl"
    if os.path.exists(pk): return pickle.load(open(pk, 'rb'))
    det = predict(key, TEST_IMGS()); pickle.dump(det, open(pk, 'wb')); return det

def model_stats(key):
    line = ''
    for l in open(f"{exp_dir(key)}/train.log", errors='ignore'):
        if 'layers' in l and 'param' in l: line = l.strip()
    return line

def eval_baseline(key):
    paths = TEST_IMGS(); gt22 = load_gt(paths); det = get_detections(key)
    gt6 = [np.c_[SUP22[g[:, 0].astype(int)], g[:, 1:]] if len(g) else g for g in gt22]
    det6 = [np.c_[d[:, :5], SUP22[d[:, 5].astype(int)]] if len(d) else d for d in det]
    res = dict(model=key, phase='baseline', detector_22class=compute_metrics(det, gt22, 22), final_6class=compute_metrics(det6, gt6, 6),
               train_minutes=json.load(open(f"{exp_dir(key)}/train_time.json"))['train_minutes'], model_summary=model_stats(key),
               eval_cfg=dict(conf=0.001, nms_iou=0.6, max_det=100, imgsz=IMG, split='test'))
    json.dump(res, open(f"{exp_dir(key)}/metrics_test.json", 'w'), indent=1)
    print(key, "22-class:", {k: round(v, 3) for k, v in res['detector_22class'].items() if k in ('P', 'R', 'mAP50', 'mAP50_95')})
    print(key, " 6-class:", {k: round(v, 3) for k, v in res['final_6class'].items() if k in ('P', 'R', 'mAP50', 'mAP50_95')})

## Phase 1 — Baselines (YOLO only). Each cell trains 100 epochs, saves logs/weights to Drive, then evaluates on the TACO test split.
Re-running a cell skips training if `best.pt` already exists on Drive (safe against Colab disconnects).

In [ ]:
# Experiment 1 — YOLOv5s
train_detector('yolov5s'); eval_baseline('yolov5s')

[skip] exp1_yolov5s already trained
yolov5s 22-class: {'P': 0.195, 'R': 0.173, 'mAP50': 0.138, 'mAP50_95': 0.095}
yolov5s  6-class: {'P': 0.301, 'R': 0.238, 'mAP50': 0.187, 'mAP50_95': 0.126}


In [ ]:
# Experiment 2 — YOLOv5l
train_detector('yolov5l'); eval_baseline('yolov5l')

[skip] exp2_yolov5l already trained
yolov5l 22-class: {'P': 0.184, 'R': 0.15, 'mAP50': 0.112, 'mAP50_95': 0.088}
yolov5l  6-class: {'P': 0.278, 'R': 0.243, 'mAP50': 0.159, 'mAP50_95': 0.119}


In [ ]:
# Experiment 3 — YOLOv8s
train_detector('yolov8s'); eval_baseline('yolov8s')

[skip] exp3_yolov8s already trained
yolov8s 22-class: {'P': 0.163, 'R': 0.221, 'mAP50': 0.18, 'mAP50_95': 0.144}
yolov8s  6-class: {'P': 0.157, 'R': 0.241, 'mAP50': 0.177, 'mAP50_95': 0.134}


In [ ]:
# Experiment 4 — YOLO26s
train_detector('yolo26s'); eval_baseline('yolo26s')

[skip] exp4_yolo26s already trained
yolo26s 22-class: {'P': 0.19, 'R': 0.312, 'mAP50': 0.186, 'mAP50_95': 0.161}
yolo26s  6-class: {'P': 0.179, 'R': 0.371, 'mAP50': 0.193, 'mAP50_95': 0.161}


In [ ]:
# CELL — train InceptionV3 on TACO crops (standalone process so TensorFlow releases the GPU afterwards)
script = r'''
import os, sys, json, tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import InceptionV3
for g in tf.config.list_physical_devices('GPU'): tf.config.experimental.set_memory_growth(g, True)
tf.keras.utils.set_random_seed(0)
CLS_DIR, OUT, EPOCHS, BS, SZ = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4]), int(sys.argv[5])
CLASSES = ['cardboard', 'paper', 'plastic', 'metal', 'glass', 'trash']
def ds(sp, shuffle):
    d = tf.keras.utils.image_dataset_from_directory(f"{CLS_DIR}/{sp}", class_names=CLASSES, label_mode='categorical',
                                                    image_size=(SZ, SZ), batch_size=BS, shuffle=shuffle, seed=0)
    return d.cache().prefetch(tf.data.AUTOTUNE)
tr, va, te = ds('train', True), ds('val', False), ds('test', False)
base = InceptionV3(include_top=False, weights='imagenet', input_shape=(SZ, SZ, 3), pooling='avg'); base.trainable = False  # transfer learning
aug = tf.keras.Sequential([layers.RandomRotation(0.1), layers.RandomContrast(0.2), layers.RandomBrightness(0.2)])        # rotation + colour jitter
inp = layers.Input((SZ, SZ, 3)); x = aug(inp); x = layers.Rescaling(1/127.5, offset=-1)(x)                               # = inception preprocess
x = base(x, training=False); out = layers.Dense(6, activation='softmax')(x)                                              # [None, 6]
model = tf.keras.Model(inp, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
cb = [tf.keras.callbacks.ModelCheckpoint(f"{OUT}/inception_best.keras", monitor='val_accuracy', save_best_only=True),
      tf.keras.callbacks.CSVLogger(f"{OUT}/history.csv")]
h = model.fit(tr, validation_data=va, epochs=EPOCHS, callbacks=cb, verbose=2)
best = tf.keras.models.load_model(f"{OUT}/inception_best.keras")
res = dict(train_acc_last_epoch=h.history['accuracy'][-1], best_val_acc=max(h.history['val_accuracy']),
           val_acc=best.evaluate(va, verbose=0)[1], test_acc=best.evaluate(te, verbose=0)[1],
           total_params=int(model.count_params()), epochs=EPOCHS, batch=BS, input=SZ)
json.dump(res, open(f"{OUT}/classifier_metrics.json", 'w'), indent=1); print(res)
'''
INC = f'{ROOT}/inception'; os.makedirs(INC, exist_ok=True)
open('/content/train_inception.py', 'w').write(script)
if not os.path.exists(f'{INC}/classifier_metrics.json'):
    run_cmd(['python', '/content/train_inception.py', CDIR, INC, str(CLS_EPOCHS), str(CLS_BATCH), str(CLS_IMG)], f'{INC}/train.log')
print(open(f'{INC}/classifier_metrics.json').read())

{
 "train_acc_last_epoch": 0.8608644008636475,
 "best_val_acc": 0.6908212304115295,
 "val_acc": 0.6908212304115295,
 "test_acc": 0.7052631378173828,
 "total_params": 21815078,
 "epochs": 100,
 "batch": 64,
 "input": 224
}


In [ ]:
# CELLS — Experiments 5-8: hybrids (YOLOv5s/YOLOv5l/YOLOv8s/YOLO26s + Inception), evaluated on the same TACO test split
import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
inc = tf.keras.models.load_model(f'{INC}/inception_best.keras')

def eval_hybrid(key):
    paths = TEST_IMGS(); gt22 = load_gt(paths); det = get_detections(key)
    gt6 = [np.c_[SUP22[g[:, 0].astype(int)], g[:, 1:]] if len(g) else g for g in gt22]
    out = []
    n_boxes = n_oob = 0                                   # sanity counters
    for p, d in zip(paths, det):
        if not len(d): out.append(np.zeros((0, 6))); continue
        keep = torchvision.ops.nms(torch.tensor(d[:, :4], dtype=torch.float32), torch.tensor(d[:, 4], dtype=torch.float32), 0.6).numpy()
        d = d[keep]                                       # class-agnostic: detector acts as a region proposer
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB); Hh, Ww = img.shape[:2]; crops = []
        n_boxes += len(d)
        n_oob += int(((d[:, 0] >= Ww) | (d[:, 1] >= Hh) | (d[:, 2] <= 0) | (d[:, 3] <= 0)).sum())
        for x1, y1, x2, y2 in d[:, :4]:
            # clamp start AND end inside the image so the crop is never empty
            x1 = int(np.clip(x1, 0, Ww - 1)); y1 = int(np.clip(y1, 0, Hh - 1))
            x2 = int(np.clip(x2, x1 + 1, Ww)); y2 = int(np.clip(y2, y1 + 1, Hh))
            crops.append(cv2.resize(img[y1:y2, x1:x2], (CLS_IMG, CLS_IMG)))
        prob = inc.predict(np.array(crops, dtype='float32'), batch_size=256, verbose=0)
        conf = d[:, 4] * (prob.max(1) if HYBRID_SCORE == 'det*cls' else 1)
        out.append(np.c_[d[:, :4], conf, prob.argmax(1)])
    if n_oob:
        print(f'WARNING {key}: {n_oob}/{n_boxes} boxes ({100*n_oob/max(n_boxes,1):.1f}%) were outside the image')
    res = dict(model=f'{key}+inception', phase='hybrid', detector=key, final_6class=compute_metrics(out, gt6, 6),
               classifier=json.load(open(f'{INC}/classifier_metrics.json')), hybrid_score=HYBRID_SCORE,
               train_minutes=json.load(open(f"{exp_dir(key)}/train_time.json"))['train_minutes'],
               eval_cfg=dict(conf=0.001, nms_iou=0.6, agnostic_nms_before_classifier=True, max_det=100, split='test'))
    os.makedirs(hyb_dir(key), exist_ok=True)
    json.dump(res, open(f'{hyb_dir(key)}/metrics_test.json', 'w'), indent=1)
    shutil.copytree(f'{exp_dir(key)}/weights', f'{hyb_dir(key)}/detector_weights', dirs_exist_ok=True)
    shutil.copy(f'{INC}/inception_best.keras', f'{hyb_dir(key)}/inception_best.keras')
    shutil.copy(f'{INC}/history.csv', f'{hyb_dir(key)}/inception_history.csv')
    print(key + '+inception 6-class:', {k: round(v, 3) for k, v in res['final_6class'].items() if k in ('P', 'R', 'mAP50', 'mAP50_95')})

for k in EXPS: eval_hybrid(k)          # experiments 5, 6, 7, 8

WARNING yolov5s: 15/3612 boxes (0.4%) were outside the image
yolov5s+inception 6-class: {'P': 0.343, 'R': 0.34, 'mAP50': 0.246, 'mAP50_95': 0.16}
WARNING yolov5l: 17/1676 boxes (1.0%) were outside the image
yolov5l+inception 6-class: {'P': 0.31, 'R': 0.327, 'mAP50': 0.254, 'mAP50_95': 0.187}
WARNING yolov8s: 1/4000 boxes (0.0%) were outside the image
yolov8s+inception 6-class: {'P': 0.325, 'R': 0.316, 'mAP50': 0.237, 'mAP50_95': 0.185}
WARNING yolo26s: 15/1636 boxes (0.9%) were outside the image
yolo26s+inception 6-class: {'P': 0.268, 'R': 0.302, 'mAP50': 0.202, 'mAP50_95': 0.163}


In [ ]:
# CELL — config audit: confirm every run used the paper's settings
import yaml
print(f"{'run':<18}{'epochs':>7}{'batch':>6}{'imgsz':>6}{'pretrained/model':>22}   summary")
for k, e in EXPS.items():
    f = f"{exp_dir(k)}/opt.yaml" if e['fam'] == 'v5' else f"{exp_dir(k)}/args.yaml"
    a = yaml.safe_load(open(f)); mdl = a.get('weights') or a.get('model')
    print(f"{k:<18}{a['epochs']:>7}{a['batch_size' if e['fam']=='v5' else 'batch']:>6}{a['imgsz']:>6}{str(mdl):>22}   {model_stats(k)}")
print("\nPaper (YOLOv5l, 22 classes): 468 layers, 46,251,379 parameters, 100 epochs, batch 16, img 416")
print("Inception:", json.load(open(f'{INC}/classifier_metrics.json')))

run                epochs batch imgsz      pretrained/model   summary
yolov5s               100    16   416            yolov5s.pt   Model summary: 68 layers, 7,069,459 parameters, 0 gradients, 15.9 GFLOPs
yolov5l               100    16   416            yolov5l.pt   Model summary: 112 layers, 46,221,363 parameters, 0 gradients, 108.0 GFLOPs
yolov8s               100    16   416            yolov8s.pt   Model summary (fused): 72 layers, 11,134,098 parameters, 0 gradients, 28.5 GFLOPs
yolo26s               100    16   416            yolo26s.pt   YOLO26s summary (fused): 120 layers, 9,473,694 parameters, 0 gradients, 20.8 GFLOPs

Paper (YOLOv5l, 22 classes): 468 layers, 46,251,379 parameters, 100 epochs, batch 16, img 416
Inception: {'train_acc_last_epoch': 0.8608644008636475, 'best_val_acc': 0.6908212304115295, 'val_acc': 0.6908212304115295, 'test_acc': 0.7052631378173828, 'total_params': 21815078, 'epochs': 100, 'batch': 64, 'input': 224}


In [ ]:
# CELL — final comparison of all 8 models (saved to Drive/comparison)
import pandas as pd, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
rows = []
for k in EXPS:
    b = json.load(open(f'{exp_dir(k)}/metrics_test.json')); h = json.load(open(f'{hyb_dir(k)}/metrics_test.json'))
    for tag, r in ((k, b), (k + ' + Inception', h)):
        m = r['final_6class']
        rows.append({'Experiment': f"exp{EXPS[k]['id'] + (0 if tag == k else 4)}", 'Model': tag, 'Phase': r['phase'],
                     'P (6-class)': m['P'], 'R (6-class)': m['R'], 'mAP@.5 (6-class)': m['mAP50'], 'mAP@.5:.95 (6-class)': m['mAP50_95'],
                     'mAP@.5 (22-class, detector)': b['detector_22class']['mAP50'], 'mAP@.5:.95 (22-class, detector)': b['detector_22class']['mAP50_95'],
                     'Detector train time (min)': b['train_minutes']})
df = pd.DataFrame(rows).sort_values('Experiment').reset_index(drop=True)
df.round(4).to_csv(f'{ROOT}/comparison/all_models_test_metrics.csv', index=False)
print(df.round(4).to_string(index=False))

lines = ["# Comparative analysis (TACO held-out test split, 6 super-categories)\n"]
for col in ['P (6-class)', 'R (6-class)', 'mAP@.5 (6-class)', 'mAP@.5:.95 (6-class)']:
    r = df.loc[df[col].idxmax()]; lines.append(f"- Best {col}: **{r['Model']}** ({r[col]:.3f})")
lines.append("\n## Effect of adding the Inception second stage (hybrid minus baseline)\n")
for k in EXPS:
    b, h = df[df.Model == k].iloc[0], df[df.Model == k + ' + Inception'].iloc[0]
    lines.append(f"- {k}: mAP@.5 {h['mAP@.5 (6-class)']-b['mAP@.5 (6-class)']:+.3f}, mAP@.5:.95 {h['mAP@.5:.95 (6-class)']-b['mAP@.5:.95 (6-class)']:+.3f}, "
                 f"P {h['P (6-class)']-b['P (6-class)']:+.3f}, R {h['R (6-class)']-b['R (6-class)']:+.3f}")
lines.append(f"\nInception classifier on TACO crops (test acc): {json.load(open(f'{INC}/classifier_metrics.json'))['test_acc']:.3f}")
open(f'{ROOT}/comparison/analysis.md', 'w').write('\n'.join(lines)); print('\n' + '\n'.join(lines))

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for a, col in zip(ax, ['mAP@.5 (6-class)', 'mAP@.5:.95 (6-class)']):
    a.bar(df.Model, df[col], color=['#4C72B0' if p == 'baseline' else '#DD8452' for p in df.Phase]); a.set_title(col)
    a.tick_params(axis='x', rotation=60)
plt.tight_layout(); plt.savefig(f'{ROOT}/comparison/map_comparison.png', dpi=200)

import re
def map50_col(cols):
    for c in cols:
        s = c.lower()
        if 'map' in s and '95' not in s and re.search(r'map[_@]?(0\.5|50)', s):
            return c
    return None

plt.figure(figsize=(8, 4.5))                      # validation mAP@.5 over epochs (training-time val split)
for k, e in EXPS.items():
    r = pd.read_csv(f"{exp_dir(k)}/results.csv"); r.columns = [c.strip() for c in r.columns]
    c = map50_col(r.columns)
    if c is None:
        print(f'skip {k}: no mAP50 column ->', list(r.columns)); continue
    x = r['epoch'] if 'epoch' in r.columns else range(len(r))
    plt.plot(x, r[c], label=k)
plt.xlabel('epoch'); plt.ylabel('val mAP@.5 (22-class)'); plt.legend(); plt.tight_layout()
plt.savefig(f'{ROOT}/comparison/val_map_curves.png', dpi=200)

Experiment               Model    Phase  P (6-class)  R (6-class)  mAP@.5 (6-class)  mAP@.5:.95 (6-class)  mAP@.5 (22-class, detector)  mAP@.5:.95 (22-class, detector)  Detector train time (min)
      exp1             yolov5s baseline       0.3009       0.2381            0.1869                0.1258                       0.1377                           0.0949                     9.7039
      exp2             yolov5l baseline       0.2780       0.2427            0.1590                0.1195                       0.1118                           0.0884                    19.4250
      exp3             yolov8s baseline       0.1569       0.2406            0.1774                0.1342                       0.1796                           0.1444                     7.3046
      exp4             yolo26s baseline       0.1795       0.3707            0.1934                0.1614                       0.1864                           0.1613                    11.8451
      exp5 yolov5s + Ince

In [ ]:
# CELL — bundle everything (optional download)
shutil.make_archive('/content/waste_experiments_results', 'zip', ROOT)
from google.colab import files; files.download('/content/waste_experiments_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
from google.colab import drive; drive.mount('/content/drive')
import glob, json
files = glob.glob('/content/drive/MyDrive/**/metrics_test.json', recursive=True)
print(len(files), 'files found')
for f in files:
    d = json.load(open(f))
    if 'detector_22class' in d:
        print(f); print('  ', list(d['detector_22class'].keys())); break

Mounted at /content/drive
8 files found
/content/drive/MyDrive/waste_experiments/exp1_yolov5s/metrics_test.json
   ['P', 'R', 'mAP50', 'mAP50_95', 'n_images', 'n_gt', 'classes_evaluated', 'per_class_AP50']


In [5]:
import glob, json, os, re, pandas as pd
R = '/content/drive/MyDrive/waste_experiments'

base, hyb = {}, {}
for f in glob.glob(f'{R}/**/metrics_test.json', recursive=True):
    d = json.load(open(f))
    if d.get('phase') == 'hybrid':
        hyb[d['detector']] = d
    else:
        m = re.match(r'exp(\d+)_(.+)', os.path.basename(os.path.dirname(f)))
        if m: base[m.group(2)] = (int(m.group(1)), d)

rows = []
for k, (i, b) in base.items():
    d22 = b['detector_22class']
    for tag, r, eid in ((k, b, i), (k + ' + Inception', hyb.get(k), i + 4)):
        if r is None: print('missing hybrid for', k); continue
        m = r['final_6class']
        rows.append({'Experiment': f'exp{eid}', 'Model': tag, 'Phase': r.get('phase', 'baseline'),
                     'P (6-class)': m['P'], 'R (6-class)': m['R'],
                     'mAP@.5 (6-class)': m['mAP50'], 'mAP@.5:.95 (6-class)': m['mAP50_95'],
                     'P (22-class, detector)': d22['P'], 'R (22-class, detector)': d22['R'],
                     'mAP@.5 (22-class, detector)': d22['mAP50'], 'mAP@.5:.95 (22-class, detector)': d22['mAP50_95'],
                     'Detector train time (min)': b['train_minutes']})

df = pd.DataFrame(rows).sort_values('Experiment', key=lambda s: s.str[3:].astype(int)).reset_index(drop=True)
os.makedirs(f'{R}/comparison', exist_ok=True)
df.round(4).to_csv(f'{R}/comparison/all_models_test_metrics.csv', index=False)
print(df.round(4).to_string(index=False))

Experiment               Model    Phase  P (6-class)  R (6-class)  mAP@.5 (6-class)  mAP@.5:.95 (6-class)  P (22-class, detector)  R (22-class, detector)  mAP@.5 (22-class, detector)  mAP@.5:.95 (22-class, detector)  Detector train time (min)
      exp1             yolov5s baseline       0.3009       0.2381            0.1869                0.1258                  0.1947                  0.1734                       0.1377                           0.0949                     9.7039
      exp2             yolov5l baseline       0.2780       0.2427            0.1590                0.1195                  0.1840                  0.1503                       0.1118                           0.0884                    19.4250
      exp3             yolov8s baseline       0.1569       0.2406            0.1774                0.1342                  0.1628                  0.2206                       0.1796                           0.1444                     7.3046
      exp4             yolo2